# Module 02-09 — Stacking & Voting Ensembles

**Learning objective:** Implement hard voting, soft voting, meta-learner stacking, and blending from scratch, then measure when diversity and combination strategy actually improve performance.

**Prerequisites:** 02-03 (Decision Trees & Random Forests), 02-04 (Gradient Boosting & AdaBoost)

---

## Part 0 — Setup & Prerequisites

Ensemble methods combine predictions from multiple base models to produce a single, more accurate output. Three combination strategies are covered:

1. **Hard voting** — majority-vote over predicted class labels
2. **Soft voting** — average the predicted class *probabilities*, then argmax
3. **Stacking** — train a *meta-learner* (level-1 model) on the out-of-fold predictions of level-0 base models
4. **Blending** — a simpler stacking variant that uses a held-out validation set instead of cross-validation

We evaluate everything on the **scikit-learn Digits dataset** (1 797 samples, 64 features, 10 classes).

In [ ]:
import sys
import time
import copy
import warnings
import random
from collections import Counter
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier as SklearnStackingClassifier,
    RandomForestClassifier,
)

import torch
warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Configuration
TEST_SIZE   = 0.20   # 80/20 train-test split (classical ML convention)
N_FOLDS     = 5      # cross-validation folds for stacking
BLEND_FRAC  = 0.20   # fraction of training set reserved as blend hold-out

In [ ]:
# Load & Inspect Dataset
digits = load_digits()
X_all, y_all = digits.data, digits.target          # (1797, 64), (1797,)

print(f"Samples : {X_all.shape[0]}")
print(f"Features: {X_all.shape[1]}  (8x8 pixel values, range 0-16)")
print(f"Classes : {len(np.unique(y_all))}  (digits 0-9)")
print(f"Class distribution: {dict(Counter(y_all))}")

# Train / test split (80 / 20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=SEED, stratify=y_all
)
print(f"\nTrain: {X_train_full.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

# Visualise a few examples
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for idx, ax in enumerate(axes.flat):
    ax.imshow(digits.images[idx], cmap="gray_r")
    ax.set_title(str(y_all[idx]), fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Digits (label shown above each image)", y=1.02)
plt.tight_layout()
plt.show()

---

## Part 1 — Voting Ensembles from Scratch

### Hard Voting

Each base classifier casts a *vote* for one class label. The ensemble selects the class with the most votes (plurality). With $M$ models and $K$ classes, for input $\mathbf{x}$:

$$\hat{y}_{\text{hard}} = \arg\max_k \sum_{m=1}^{M} \mathbf{1}[\hat{y}^{(m)} = k]$$

### Soft Voting

Each model outputs a probability vector $\hat{\mathbf{p}}^{(m)} \in \mathbb{R}^K$. The ensemble averages these vectors (or weights them), then picks the most probable class:

$$\hat{y}_{\text{soft}} = \arg\max_k \frac{1}{M} \sum_{m=1}^{M} \hat{p}_k^{(m)}$$

Soft voting is generally superior because it accounts for *confidence*, not just which class wins.

> **When does voting help?** When base models make *different* errors — i.e., they are *diverse*. If all models fail on the same inputs, majority vote cannot rescue them.

In [ ]:
class HardVotingClassifier:
    """Hard majority-vote ensemble of heterogeneous classifiers.

    Attributes:
        estimators: List of (name, fitted_classifier) pairs.
    """

    def __init__(self, estimators: list) -> None:
        """Initialise with a list of (name, classifier) pairs.

        Args:
            estimators: Each element is (name, sklearn-compatible classifier).
        """
        self.estimators = estimators

    def fit(self, X: np.ndarray, y: np.ndarray) -> "HardVotingClassifier":
        """Fit all base estimators.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Target labels of shape (n_samples,).

        Returns:
            self
        """
        for _, clf in self.estimators:
            clf.fit(X, y)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict by majority vote.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,).
        """
        # collect predictions: shape (n_estimators, n_samples)
        all_preds = np.array([clf.predict(X) for _, clf in self.estimators])
        # majority vote per sample
        predictions = np.apply_along_axis(
            lambda col: Counter(col).most_common(1)[0][0],
            axis=0,
            arr=all_preds,
        )
        return predictions

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on (X, y).

        Args:
            X: Feature matrix.
            y: True labels.

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

### Soft Voting Implementation

Soft voting requires each base model to expose `predict_proba`. We average the probability matrices — one per model — and pick the column with the highest mean probability.

In [ ]:
class SoftVotingClassifier:
    """Probability-averaging (soft-vote) ensemble of heterogeneous classifiers.

    Attributes:
        estimators: List of (name, fitted_classifier) pairs.
        weights: Optional per-model weights for the probability average.
    """

    def __init__(
        self,
        estimators: list,
        weights: list | None = None,
    ) -> None:
        """Initialise with estimators and optional weights.

        Args:
            estimators: Each element is (name, sklearn-compatible classifier
                        with predict_proba).
            weights: Importance weight per model. If None, equal weights used.
        """
        self.estimators = estimators
        self.weights = weights

    def fit(self, X: np.ndarray, y: np.ndarray) -> "SoftVotingClassifier":
        """Fit all base estimators.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Target labels of shape (n_samples,).

        Returns:
            self
        """
        self.classes_ = np.unique(y)
        for _, clf in self.estimators:
            clf.fit(X, y)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return weighted-average probability matrix.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Probability matrix of shape (n_samples, n_classes).
        """
        # shape: (n_estimators, n_samples, n_classes)
        proba_list = np.array([clf.predict_proba(X) for _, clf in self.estimators])
        if self.weights is not None:
            w = np.array(self.weights, dtype=float)
            w = w / w.sum()
            avg_proba = np.average(proba_list, axis=0, weights=w)
        else:
            avg_proba = proba_list.mean(axis=0)
        return avg_proba

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict by averaging probabilities.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,).
        """
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on (X, y).

        Args:
            X: Feature matrix.
            y: True labels.

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

---

## Part 2 — Stacking & Blending from Scratch

### Stacking (Level-0 → Level-1)

Stacking trains a *meta-learner* on the predictions of base models. The key challenge: if we train the base models on the same data used to generate their predictions for the meta-learner, the meta-learner trains on *over-fitted* predictions — causing data leakage.

**Solution: out-of-fold (OOF) predictions via $k$-fold cross-validation.**

Algorithm:
1. Split training data into $k$ folds.
2. For each fold $i$: train all base models on the other $k-1$ folds, then predict on fold $i$ → these are *out-of-fold* (OOF) predictions.
3. After iterating all folds, every training sample has one OOF prediction from each base model.
4. Stack the OOF predictions as a new feature matrix $\tilde{X}_{\text{train}}$ (shape: $n_{\text{train}} \times M$ for $M$ base models).
5. Retrain each base model on **all** training data.
6. For test data: use the fully-retrained base models to generate test meta-features.
7. Train the meta-learner on $\tilde{X}_{\text{train}}$, then predict on $\tilde{X}_{\text{test}}$.

### Blending (Simpler Alternative)

Instead of cross-validation, reserve a *hold-out blend set* from the training data:

1. Split training → `train_base` + `blend_hold_out`.
2. Fit base models on `train_base`.
3. Predict on `blend_hold_out` → `blend_features`.
4. Train meta-learner on `blend_features` (targets = `y_blend_hold_out`).
5. Predict test with base models + meta-learner directly.

Blending wastes data (hold-out is unused for base model training) but is faster and simpler to implement.

In [ ]:
class StackingEnsemble:
    """Meta-learner stacking with out-of-fold cross-validation.

    Level-0 base models produce out-of-fold predictions which become
    the feature matrix for the level-1 meta-learner, preventing data leakage.

    Attributes:
        base_estimators: List of (name, classifier) pairs for level-0.
        meta_estimator: Level-1 classifier that combines base predictions.
        n_folds: Number of cross-validation folds for OOF generation.
        use_proba: If True, stack class probabilities; otherwise stack labels.
        classes_: Unique class labels observed during fit.
    """

    def __init__(
        self,
        base_estimators: list,
        meta_estimator: object,
        n_folds: int = 5,
        use_proba: bool = True,
    ) -> None:
        """Initialise the stacking ensemble.

        Args:
            base_estimators: Level-0 classifiers as (name, clf) pairs.
            meta_estimator: Level-1 meta-learner.
            n_folds: Cross-validation folds for out-of-fold generation.
            use_proba: Stack probabilities (True) or hard labels (False).
        """
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator
        self.n_folds = n_folds
        self.use_proba = use_proba

    def _oof_predictions(
        self,
        X: np.ndarray,
        y: np.ndarray,
        clf: object,
    ) -> np.ndarray:
        """Generate out-of-fold predictions for one base estimator.

        Args:
            X: Full training feature matrix.
            y: Full training labels.
            clf: Unfitted classifier (will be deep-copied and re-fitted per fold).

        Returns:
            OOF prediction array. Shape (n_samples, n_classes) if use_proba,
            else (n_samples,).
        """
        n_classes = len(np.unique(y))
        skf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=SEED)

        if self.use_proba:
            oof = np.zeros((len(X), n_classes))
        else:
            oof = np.zeros(len(X))

        for train_idx, val_idx in skf.split(X, y):
            X_fold_tr, X_fold_val = X[train_idx], X[val_idx]
            y_fold_tr = y[train_idx]

            clf_copy = copy.deepcopy(clf)
            clf_copy.fit(X_fold_tr, y_fold_tr)

            if self.use_proba:
                oof[val_idx] = clf_copy.predict_proba(X_fold_val)
            else:
                oof[val_idx] = clf_copy.predict(X_fold_val)

        return oof

    def fit(self, X: np.ndarray, y: np.ndarray) -> "StackingEnsemble":
        """Fit stacking ensemble using out-of-fold cross-validation.

        Generates OOF predictions for each base model, assembles meta-features,
        retrains base models on full training data, then trains meta-learner.

        Args:
            X: Training feature matrix of shape (n_samples, n_features).
            y: Training labels of shape (n_samples,).

        Returns:
            self
        """
        self.classes_ = np.unique(y)

        # Step 1: collect OOF predictions from each base model
        oof_columns = []
        for name, clf in self.base_estimators:
            oof = self._oof_predictions(X, y, clf)
            oof_columns.append(oof)
            print(f"  OOF generated for {name}, shape={oof.shape}")

        # Step 2: build meta-feature matrix (n_samples, total_meta_features)
        meta_X_train = np.hstack(
            [col.reshape(len(X), -1) for col in oof_columns]
        )

        # Step 3: re-fit base models on the FULL training set
        for name, clf in self.base_estimators:
            clf.fit(X, y)

        # Step 4: train meta-learner on OOF features
        self.meta_estimator.fit(meta_X_train, y)
        print(f"  Meta-learner trained on meta-features shape {meta_X_train.shape}")
        return self

    def _build_test_meta_features(self, X_test: np.ndarray) -> np.ndarray:
        """Generate meta-features for the test set from fully-fitted base models.

        Args:
            X_test: Test feature matrix of shape (n_samples, n_features).

        Returns:
            Meta-feature matrix of shape (n_test, total_meta_features).
        """
        columns = []
        for _, clf in self.base_estimators:
            if self.use_proba:
                preds = clf.predict_proba(X_test)
            else:
                preds = clf.predict(X_test).reshape(-1, 1)
            columns.append(preds.reshape(len(X_test), -1))
        return np.hstack(columns)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict labels using the fitted stacking ensemble.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,).
        """
        meta_X = self._build_test_meta_features(X)
        return self.meta_estimator.predict(meta_X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities using the fitted stacking ensemble.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Probability matrix of shape (n_samples, n_classes).
        """
        meta_X = self._build_test_meta_features(X)
        return self.meta_estimator.predict_proba(meta_X)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on (X, y).

        Args:
            X: Feature matrix.
            y: True labels.

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

### Blending Classifier

Blending avoids cross-validation by reserving a separate *blend hold-out* set. This is faster but wastes data that could otherwise train the base models.

In [ ]:
class BlendingEnsemble:
    """Blending ensemble: base models trained on subset, meta-learner on hold-out.

    Unlike stacking, blending reserves a hold-out blend set from training data
    to generate meta-features, rather than using cross-validation.

    Attributes:
        base_estimators: List of (name, classifier) pairs for level-0.
        meta_estimator: Level-1 classifier that combines base predictions.
        blend_size: Fraction of training data reserved as blend hold-out.
        use_proba: If True, stack class probabilities; otherwise stack labels.
        classes_: Unique class labels observed during fit.
    """

    def __init__(
        self,
        base_estimators: list,
        meta_estimator: object,
        blend_size: float = 0.2,
        use_proba: bool = True,
    ) -> None:
        """Initialise the blending ensemble.

        Args:
            base_estimators: Level-0 classifiers as (name, clf) pairs.
            meta_estimator: Level-1 meta-learner.
            blend_size: Fraction held-out for meta-learner training.
            use_proba: Stack probabilities (True) or hard labels (False).
        """
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator
        self.blend_size = blend_size
        self.use_proba = use_proba

    def fit(self, X: np.ndarray, y: np.ndarray) -> "BlendingEnsemble":
        """Fit blending ensemble.

        Splits training data into base-training set and blend hold-out.
        Base models train on the base-training set; their predictions on the
        hold-out become meta-features for the meta-learner.

        Args:
            X: Training feature matrix of shape (n_samples, n_features).
            y: Training labels of shape (n_samples,).

        Returns:
            self
        """
        self.classes_ = np.unique(y)

        X_base_tr, X_blend, y_base_tr, y_blend = train_test_split(
            X, y,
            test_size=self.blend_size,
            random_state=SEED,
            stratify=y,
        )
        print(f"  Base-train: {X_base_tr.shape[0]}  |  Blend hold-out: {X_blend.shape[0]}")

        blend_columns = []
        for name, clf in self.base_estimators:
            clf.fit(X_base_tr, y_base_tr)
            if self.use_proba:
                preds = clf.predict_proba(X_blend)
            else:
                preds = clf.predict(X_blend).reshape(-1, 1)
            blend_columns.append(preds.reshape(len(X_blend), -1))
            print(f"  Blend features from {name}: shape {preds.shape}")

        meta_X_blend = np.hstack(blend_columns)
        self.meta_estimator.fit(meta_X_blend, y_blend)
        print(f"  Meta-learner trained on blend features {meta_X_blend.shape}")
        return self

    def _build_test_meta_features(self, X_test: np.ndarray) -> np.ndarray:
        """Generate meta-features for the test set from base estimators.

        Args:
            X_test: Test feature matrix.

        Returns:
            Meta-feature matrix of shape (n_test, total_meta_features).
        """
        columns = []
        for _, clf in self.base_estimators:
            if self.use_proba:
                preds = clf.predict_proba(X_test)
            else:
                preds = clf.predict(X_test).reshape(-1, 1)
            columns.append(preds.reshape(len(X_test), -1))
        return np.hstack(columns)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict labels using the fitted blending ensemble.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,).
        """
        meta_X = self._build_test_meta_features(X)
        return self.meta_estimator.predict(meta_X)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on (X, y).

        Args:
            X: Feature matrix.
            y: True labels.

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

---

## Part 3 — Training on Digits

We define a diverse set of base models, then train and compare all four ensemble strategies: hard voting, soft voting, stacking, and blending. Diversity is critical — if all base models are identical, combining them cannot improve performance.

### Base Model Selection

We choose classifiers with different inductive biases:
- **Decision Tree** — high variance, interpretable splits
- **k-NN (k=5)** — distance-based, non-parametric
- **Gaussian Naive Bayes** — probabilistic, assumes feature independence
- **Logistic Regression** — linear decision boundary, well-calibrated probabilities

Each operates on a different region of the bias-variance spectrum, maximising diversity.

In [ ]:
# Scale features — KNN and LR are sensitive to feature scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_full)
X_test_sc  = scaler.transform(X_test)


def make_base_estimators() -> list:
    """Create a fresh set of diverse base estimators.

    Returns:
        List of (name, unfitted classifier) pairs with different inductive biases.
    """
    return [
        ("DTree",  DecisionTreeClassifier(max_depth=8, random_state=SEED)),
        ("KNN",    KNeighborsClassifier(n_neighbors=5)),
        ("GNB",    GaussianNB()),
        ("LogReg", LogisticRegression(max_iter=500, random_state=SEED)),
    ]


# Individual base model performance (baselines)
print("=" * 55)
print("INDIVIDUAL BASE MODEL PERFORMANCE")
print("=" * 55)
base_times: dict = {}
for name, clf in make_base_estimators():
    t0 = time.perf_counter()
    clf.fit(X_train_sc, y_train_full)
    base_times[name] = time.perf_counter() - t0
    acc = accuracy_score(y_test, clf.predict(X_test_sc))
    f1  = f1_score(y_test, clf.predict(X_test_sc), average="macro")
    print(f"  {name:<8} | Acc: {acc:.4f} | F1: {f1:.4f} | Time: {base_times[name]:.2f}s")

In [ ]:
# Hard Voting
print("\n" + "=" * 55)
print("HARD VOTING ENSEMBLE")
print("=" * 55)

hard_voter = HardVotingClassifier(estimators=make_base_estimators())
t0 = time.perf_counter()
hard_voter.fit(X_train_sc, y_train_full)
hard_time = time.perf_counter() - t0

hard_preds = hard_voter.predict(X_test_sc)
hard_acc = hard_voter.score(X_test_sc, y_test)
hard_f1  = f1_score(y_test, hard_preds, average="macro")
print(f"  Hard Vote  | Acc: {hard_acc:.4f} | F1: {hard_f1:.4f} | Time: {hard_time:.2f}s")

In [ ]:
# Soft Voting
print("\n" + "=" * 55)
print("SOFT VOTING ENSEMBLE")
print("=" * 55)

soft_voter = SoftVotingClassifier(estimators=make_base_estimators())
t0 = time.perf_counter()
soft_voter.fit(X_train_sc, y_train_full)
soft_time = time.perf_counter() - t0

soft_preds = soft_voter.predict(X_test_sc)
soft_acc = soft_voter.score(X_test_sc, y_test)
soft_f1  = f1_score(y_test, soft_preds, average="macro")
print(f"  Soft Vote  | Acc: {soft_acc:.4f} | F1: {soft_f1:.4f} | Time: {soft_time:.2f}s")

In [ ]:
# Stacking (out-of-fold, 5-fold CV)
print("\n" + "=" * 55)
print("STACKING ENSEMBLE (out-of-fold, 5-fold CV)")
print("=" * 55)

meta_lr = LogisticRegression(max_iter=1000, random_state=SEED)
stacker = StackingEnsemble(
    base_estimators=make_base_estimators(),
    meta_estimator=meta_lr,
    n_folds=N_FOLDS,
    use_proba=True,
)
t0 = time.perf_counter()
stacker.fit(X_train_sc, y_train_full)
stack_time = time.perf_counter() - t0

stack_preds = stacker.predict(X_test_sc)
stack_acc = stacker.score(X_test_sc, y_test)
stack_f1  = f1_score(y_test, stack_preds, average="macro")
print(f"  Stacking   | Acc: {stack_acc:.4f} | F1: {stack_f1:.4f} | Time: {stack_time:.2f}s")

In [ ]:
# Blending (20% hold-out)
print("\n" + "=" * 55)
print("BLENDING ENSEMBLE (20% hold-out)")
print("=" * 55)

meta_lr_blend = LogisticRegression(max_iter=1000, random_state=SEED)
blender = BlendingEnsemble(
    base_estimators=make_base_estimators(),
    meta_estimator=meta_lr_blend,
    blend_size=BLEND_FRAC,
    use_proba=True,
)
t0 = time.perf_counter()
blender.fit(X_train_sc, y_train_full)
blend_time = time.perf_counter() - t0

blend_preds = blender.predict(X_test_sc)
blend_acc = blender.score(X_test_sc, y_test)
blend_f1  = f1_score(y_test, blend_preds, average="macro")
print(f"  Blending   | Acc: {blend_acc:.4f} | F1: {blend_f1:.4f} | Time: {blend_time:.2f}s")

---

## Part 4 — Evaluation & Analysis

### 4.1 Comprehensive Results Table

We compile all results into a single DataFrame ranked by test accuracy.

In [ ]:
# Build results table
# Re-fit base models cleanly for table
base_accs: dict = {}
base_f1s:  dict = {}
for name, clf in make_base_estimators():
    clf.fit(X_train_sc, y_train_full)
    base_accs[name] = accuracy_score(y_test, clf.predict(X_test_sc))
    base_f1s[name]  = f1_score(y_test, clf.predict(X_test_sc), average="macro")

rows = []
for name in ["DTree", "KNN", "GNB", "LogReg"]:
    rows.append({
        "Model": name, "Type": "Base",
        "Acc": base_accs[name], "F1 Macro": base_f1s[name],
        "Train Time (s)": base_times.get(name, float("nan")),
    })
rows += [
    {"Model": "Hard Voting", "Type": "Ensemble", "Acc": hard_acc,
     "F1 Macro": hard_f1, "Train Time (s)": hard_time},
    {"Model": "Soft Voting", "Type": "Ensemble", "Acc": soft_acc,
     "F1 Macro": soft_f1, "Train Time (s)": soft_time},
    {"Model": "Stacking",    "Type": "Ensemble", "Acc": stack_acc,
     "F1 Macro": stack_f1, "Train Time (s)": stack_time},
    {"Model": "Blending",    "Type": "Ensemble", "Acc": blend_acc,
     "F1 Macro": blend_f1, "Train Time (s)": blend_time},
]

results_df = (
    pd.DataFrame(rows)
    .sort_values("Acc", ascending=False)
    .reset_index(drop=True)
)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Bar-chart comparison
models_list = results_df["Model"].tolist()
accs_list   = results_df["Acc"].tolist()
types_list  = results_df["Type"].tolist()
bar_colors  = ["#4472C4" if t == "Ensemble" else "#9DC3E6" for t in types_list]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models_list, accs_list, color=bar_colors, edgecolor="black", linewidth=0.6)
ax.set_ylim(min(accs_list) - 0.03, 1.01)
ax.set_ylabel("Test Accuracy")
ax.set_title("Ensemble vs Individual Model Accuracy — Digits Dataset")
ax.tick_params(axis="x", rotation=20)

for bar, acc in zip(bars, accs_list):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{acc:.3f}",
        ha="center", va="bottom", fontsize=9,
    )

legend_handles = [
    mpatches.Patch(color="#4472C4", label="Ensemble"),
    mpatches.Patch(color="#9DC3E6", label="Base model"),
]
ax.legend(handles=legend_handles)
plt.tight_layout()
plt.show()

### 4.2 Validation Against scikit-learn

We verify our from-scratch implementations produce results within expected tolerance of scikit-learn's optimised `VotingClassifier` and `StackingClassifier`.

In [ ]:
# sklearn validation
sk_hard = VotingClassifier(estimators=make_base_estimators(), voting="hard")
sk_hard.fit(X_train_sc, y_train_full)
sk_hard_acc = accuracy_score(y_test, sk_hard.predict(X_test_sc))

sk_soft = VotingClassifier(estimators=make_base_estimators(), voting="soft")
sk_soft.fit(X_train_sc, y_train_full)
sk_soft_acc = accuracy_score(y_test, sk_soft.predict(X_test_sc))

sk_stack = SklearnStackingClassifier(
    estimators=make_base_estimators(),
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=N_FOLDS,
    passthrough=False,
)
sk_stack.fit(X_train_sc, y_train_full)
sk_stack_acc = accuracy_score(y_test, sk_stack.predict(X_test_sc))

print("Validation against scikit-learn implementations")
print(f"  Hard Voting — scratch: {hard_acc:.4f}  sklearn: {sk_hard_acc:.4f}  diff: {abs(hard_acc - sk_hard_acc):.4f}")
print(f"  Soft Voting — scratch: {soft_acc:.4f}  sklearn: {sk_soft_acc:.4f}  diff: {abs(soft_acc - sk_soft_acc):.4f}")
print(f"  Stacking    — scratch: {stack_acc:.4f}  sklearn: {sk_stack_acc:.4f}  diff: {abs(stack_acc - sk_stack_acc):.4f}")
print("\n(Small stacking differences are expected due to CV fold ordering.)")

### 4.3 Diversity Analysis — Pairwise Disagreement

Two models are *diverse* if they disagree on different test samples. We quantify pairwise disagreement rate and visualise it as a heatmap. High disagreement → high diversity → better ensemble potential.

In [ ]:
def disagreement_rate(preds_a: np.ndarray, preds_b: np.ndarray) -> float:
    """Compute the fraction of samples where two classifiers disagree.

    Args:
        preds_a: Predicted labels from classifier A, shape (n_samples,).
        preds_b: Predicted labels from classifier B, shape (n_samples,).

    Returns:
        Disagreement rate in [0, 1].
    """
    return float(np.mean(preds_a != preds_b))


# Fit and collect predictions from all base models
base_preds: dict = {}
fitted_base = make_base_estimators()
for name, clf in fitted_base:
    clf.fit(X_train_sc, y_train_full)
    base_preds[name] = clf.predict(X_test_sc)

base_names_list = [n for n, _ in fitted_base]
n_base = len(base_names_list)
disagree_matrix = np.zeros((n_base, n_base))
for i, ni in enumerate(base_names_list):
    for j, nj in enumerate(base_names_list):
        disagree_matrix[i, j] = disagreement_rate(base_preds[ni], base_preds[nj])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(disagree_matrix, cmap="YlOrRd", vmin=0, vmax=0.35)
ax.set_xticks(range(n_base))
ax.set_yticks(range(n_base))
ax.set_xticklabels(base_names_list)
ax.set_yticklabels(base_names_list)
plt.colorbar(im, ax=ax, label="Disagreement Rate")
ax.set_title("Pairwise Disagreement Between Base Models")
for i in range(n_base):
    for j in range(n_base):
        ax.text(j, i, f"{disagree_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if disagree_matrix[i, j] > 0.2 else "black")
plt.tight_layout()
plt.show()

avg_disagree = disagree_matrix[np.triu_indices(n_base, k=1)].mean()
print(f"Mean pairwise disagreement: {avg_disagree:.3f}")
print("Higher disagreement = more diverse pair = greater ensemble potential.")

### 4.4 Weighted Soft Voting — Does Accuracy-Proportional Weighting Help?

We assign each base model a weight proportional to its validation accuracy, giving stronger models more influence in the probability average.

In [ ]:
def weighted_soft_vote_experiment(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    """Compare equal, accuracy-proportional, and squared-accuracy weights.

    Args:
        X_train: Scaled training features.
        y_train: Training labels.
        X_test: Scaled test features.
        y_test: True test labels.

    Returns:
        Dict mapping weight-scheme name to test accuracy.
    """
    # Estimate individual accuracy via a 15% hold-out from train
    X_tr2, X_v2, y_tr2, y_v2 = train_test_split(
        X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
    )
    val_accs_list = []
    print("Base model validation accuracies (for weighting):")
    for name, clf in make_base_estimators():
        clf.fit(X_tr2, y_tr2)
        va = accuracy_score(y_v2, clf.predict(X_v2))
        val_accs_list.append(va)
        print(f"  {name}: {va:.4f}")

    # Equal-weight
    equal_clf = SoftVotingClassifier(estimators=make_base_estimators())
    equal_clf.fit(X_train, y_train)
    equal_acc = equal_clf.score(X_test, y_test)

    # Accuracy-proportional weights
    acc_clf = SoftVotingClassifier(
        estimators=make_base_estimators(), weights=val_accs_list
    )
    acc_clf.fit(X_train, y_train)
    acc_wt_acc = acc_clf.score(X_test, y_test)

    # Squared-accuracy weights (sharper concentration)
    sq_clf = SoftVotingClassifier(
        estimators=make_base_estimators(),
        weights=[a ** 2 for a in val_accs_list],
    )
    sq_clf.fit(X_train, y_train)
    sq_wt_acc = sq_clf.score(X_test, y_test)

    print(f"\nEqual weights       : {equal_acc:.4f}")
    print(f"Accuracy weights    : {acc_wt_acc:.4f}")
    print(f"Squared-acc weights : {sq_wt_acc:.4f}")
    return {"equal": equal_acc, "accuracy": acc_wt_acc, "squared": sq_wt_acc}


weight_results = weighted_soft_vote_experiment(X_train_sc, y_train_full, X_test_sc, y_test)

### 4.5 Confusion Matrix for Best Ensemble

We examine where the stacking ensemble still makes mistakes — understanding failure modes reveals which digit pairs are hardest to separate.

In [ ]:
cm = confusion_matrix(y_test, stack_preds)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix — Stacking Ensemble (Digits)")
thresh = cm.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]),
                ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

# Top confused pairs
errors = sorted(
    [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j and cm[i, j] > 0],
    reverse=True,
)
print("Top-5 most confused pairs  (true -> predicted, count):")
for count, true_lbl, pred_lbl in errors[:5]:
    print(f"  {true_lbl} -> {pred_lbl}: {count} errors")

### 4.6 Ensemble Size Experiment

We test how accuracy changes as we add base models one by one, from best to worst individual performer. This reveals the diminishing-returns curve typical of ensemble methods.

In [ ]:
def ensemble_size_experiment(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> tuple:
    """Measure soft-vote accuracy as ensemble size grows from 1 to max models.

    Adds base models in descending order of individual test accuracy.

    Args:
        X_train: Scaled training features.
        y_train: Training labels.
        X_test: Scaled test features.
        y_test: True test labels.

    Returns:
        Tuple of (sizes: list[int], accuracies: list[float]).
    """
    named_clfs = make_base_estimators()
    for _, clf in named_clfs:
        clf.fit(X_train, y_train)
    ranked = sorted(
        named_clfs,
        key=lambda nc: accuracy_score(y_test, nc[1].predict(X_test)),
        reverse=True,
    )

    sizes: list = []
    accs:  list = []
    for k in range(1, len(ranked) + 1):
        subset = ranked[:k]
        voter = SoftVotingClassifier(estimators=subset)
        voter.fit(X_train, y_train)
        acc = voter.score(X_test, y_test)
        sizes.append(k)
        accs.append(acc)
        names_used = [n for n, _ in subset]
        print(f"  Size {k}: {names_used}  -> acc={acc:.4f}")
    return sizes, accs


sizes, ens_accs = ensemble_size_experiment(X_train_sc, y_train_full, X_test_sc, y_test)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sizes, ens_accs, marker="o", linewidth=2, color="#3A7DBF")
ax.set_xlabel("Number of base models in ensemble")
ax.set_ylabel("Test Accuracy (soft vote)")
ax.set_title("Ensemble Size vs Accuracy")
ax.set_xticks(sizes)
y_lo = min(ens_accs) - 0.02
ax.set_ylim(max(0.85, y_lo), 1.01)
plt.tight_layout()
plt.show()

### 4.7 Homogeneous Ensembles — When Diversity Is Missing

If we replace diverse base models with multiple copies of the *same* model, the ensemble gain collapses because all models make identical errors.

In [ ]:
def homogeneous_vs_diverse(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_copies: int = 4,
) -> None:
    """Compare a homogeneous ensemble against a diverse one.

    Demonstrates that soft voting on identical models yields no improvement
    over a single model, while diverse models do improve.

    Args:
        X_train: Scaled training features.
        y_train: Training labels.
        X_test: Scaled test features.
        y_test: True test labels.
        n_copies: Number of identical model copies in the homogeneous ensemble.
    """
    # Single Decision Tree baseline
    single_dt = DecisionTreeClassifier(max_depth=8, random_state=SEED)
    single_dt.fit(X_train, y_train)
    single_acc = accuracy_score(y_test, single_dt.predict(X_test))

    # Homogeneous: n copies of the same Decision Tree
    homogeneous = [
        (f"DTree_{i}", DecisionTreeClassifier(max_depth=8, random_state=SEED))
        for i in range(n_copies)
    ]
    hom_voter = SoftVotingClassifier(estimators=homogeneous)
    hom_voter.fit(X_train, y_train)
    hom_acc = hom_voter.score(X_test, y_test)

    # Diverse: four different model families
    diverse_voter = SoftVotingClassifier(estimators=make_base_estimators())
    diverse_voter.fit(X_train, y_train)
    diverse_acc = diverse_voter.score(X_test, y_test)

    print("Homogeneous vs Diverse ensemble")
    print(f"  Single DTree             : {single_acc:.4f}")
    print(f"  {n_copies}x DTree (homogeneous): {hom_acc:.4f}  (delta {hom_acc - single_acc:+.4f})")
    print(f"  Diverse 4-model ensemble : {diverse_acc:.4f}  (delta {diverse_acc - single_acc:+.4f})")
    print("\nConclusion: ensemble gain requires model DIVERSITY, not quantity alone.")


homogeneous_vs_diverse(X_train_sc, y_train_full, X_test_sc, y_test)

### 4.8 Meta-Learner Ablation — Which Level-1 Combiner Works Best?

We test different meta-learners on the same OOF features to identify the best combination strategy.

In [ ]:
def meta_learner_ablation(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> pd.DataFrame:
    """Compare meta-learner choices for the stacking level-1 combiner.

    Args:
        X_train: Scaled training features.
        y_train: Training labels.
        X_test: Scaled test features.
        y_test: True test labels.

    Returns:
        DataFrame with columns [Meta-Learner, Acc, F1 Macro, Time (s)].
    """
    meta_candidates = [
        ("LogReg",  LogisticRegression(max_iter=1000, random_state=SEED)),
        ("KNN-3",   KNeighborsClassifier(n_neighbors=3)),
        ("DTree",   DecisionTreeClassifier(max_depth=4, random_state=SEED)),
        ("GNB",     GaussianNB()),
    ]
    rows = []
    for meta_name, meta_clf in meta_candidates:
        stk = StackingEnsemble(
            base_estimators=make_base_estimators(),
            meta_estimator=meta_clf,
            n_folds=N_FOLDS,
            use_proba=True,
        )
        t0 = time.perf_counter()
        stk.fit(X_train, y_train)
        elapsed = time.perf_counter() - t0
        acc = stk.score(X_test, y_test)
        f1  = f1_score(y_test, stk.predict(X_test), average="macro")
        rows.append({"Meta-Learner": meta_name, "Acc": acc,
                     "F1 Macro": f1, "Time (s)": elapsed})
        print(f"  {meta_name:<10} | Acc: {acc:.4f} | F1: {f1:.4f} | {elapsed:.1f}s")

    return (
        pd.DataFrame(rows)
        .sort_values("Acc", ascending=False)
        .reset_index(drop=True)
    )


print("Meta-Learner Ablation (same base models, different level-1 combiner):")
ablation_df = meta_learner_ablation(X_train_sc, y_train_full, X_test_sc, y_test)
print()
print(ablation_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

### 4.9 Passthrough Stacking — Including Original Features

A popular variant passes the *original* feature matrix alongside the OOF predictions to the meta-learner. This gives the level-1 model access to raw signal that the base models may have compressed or discarded.

In [ ]:
class PassthroughStackingEnsemble:
    """Stacking ensemble that appends original features to OOF predictions.

    Extends StackingEnsemble by concatenating the original training features
    with the out-of-fold meta-features before fitting the meta-learner.

    Attributes:
        base_estimators: List of (name, classifier) pairs for level-0.
        meta_estimator: Level-1 classifier.
        n_folds: Number of stratified CV folds.
        use_proba: Stack probabilities (True) or hard labels (False).
        classes_: Unique class labels observed during fit.
    """

    def __init__(
        self,
        base_estimators: list,
        meta_estimator: object,
        n_folds: int = 5,
        use_proba: bool = True,
    ) -> None:
        """Initialise passthrough stacking ensemble.

        Args:
            base_estimators: Level-0 classifiers as (name, clf) pairs.
            meta_estimator: Level-1 meta-learner.
            n_folds: Cross-validation folds for out-of-fold generation.
            use_proba: Stack probabilities (True) or hard labels (False).
        """
        self.base_estimators = base_estimators
        self.meta_estimator  = meta_estimator
        self.n_folds         = n_folds
        self.use_proba       = use_proba
        self._inner          = StackingEnsemble(
            base_estimators=base_estimators,
            meta_estimator=meta_estimator,
            n_folds=n_folds,
            use_proba=use_proba,
        )

    def _oof_only(
        self, X: np.ndarray, y: np.ndarray
    ) -> tuple:
        """Generate raw OOF predictions without touching the meta-learner.

        Args:
            X: Training feature matrix.
            y: Training labels.

        Returns:
            Tuple of (meta_X_oof, n_classes) where meta_X_oof has shape
            (n_samples, sum_of_oof_widths).
        """
        n_classes = len(np.unique(y))
        skf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=SEED)
        oof_columns = []
        for _, clf in self.base_estimators:
            if self.use_proba:
                oof = np.zeros((len(X), n_classes))
            else:
                oof = np.zeros(len(X))
            for train_idx, val_idx in skf.split(X, y):
                clf_copy = copy.deepcopy(clf)
                clf_copy.fit(X[train_idx], y[train_idx])
                if self.use_proba:
                    oof[val_idx] = clf_copy.predict_proba(X[val_idx])
                else:
                    oof[val_idx] = clf_copy.predict(X[val_idx])
            oof_columns.append(oof.reshape(len(X), -1))
        return np.hstack(oof_columns), n_classes

    def fit(self, X: np.ndarray, y: np.ndarray) -> "PassthroughStackingEnsemble":
        """Fit using OOF predictions concatenated with original features.

        Args:
            X: Training feature matrix of shape (n_samples, n_features).
            y: Training labels of shape (n_samples,).

        Returns:
            self
        """
        self.classes_ = np.unique(y)
        oof_meta, _ = self._oof_only(X, y)
        # Passthrough: concatenate original features + OOF meta-features
        meta_X_passthrough = np.hstack([X, oof_meta])
        print(f"  Passthrough meta shape: {meta_X_passthrough.shape} "
              f"(original {X.shape[1]} + oof {oof_meta.shape[1]})")
        # Re-fit base models on full training data
        for _, clf in self.base_estimators:
            clf.fit(X, y)
        # Train meta-learner on augmented features
        self.meta_estimator.fit(meta_X_passthrough, y)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict using passthrough meta-features.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted labels of shape (n_samples,).
        """
        oof_cols = []
        for _, clf in self.base_estimators:
            if self.use_proba:
                preds = clf.predict_proba(X)
            else:
                preds = clf.predict(X).reshape(-1, 1)
            oof_cols.append(preds.reshape(len(X), -1))
        meta_X = np.hstack([X, np.hstack(oof_cols)])
        return self.meta_estimator.predict(meta_X)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on (X, y).

        Args:
            X: Feature matrix.
            y: True labels.

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

In [ ]:
# Train and evaluate passthrough stacking
print("Passthrough Stacking (original features + OOF probabilities):")
meta_lr_pt = LogisticRegression(max_iter=1000, random_state=SEED)
pt_stacker = PassthroughStackingEnsemble(
    base_estimators=make_base_estimators(),
    meta_estimator=meta_lr_pt,
    n_folds=N_FOLDS,
    use_proba=True,
)
t0 = time.perf_counter()
pt_stacker.fit(X_train_sc, y_train_full)
pt_time = time.perf_counter() - t0

pt_acc = pt_stacker.score(X_test_sc, y_test)
pt_f1  = f1_score(y_test, pt_stacker.predict(X_test_sc), average="macro")
print(f"  Passthrough Stacking | Acc: {pt_acc:.4f} | F1: {pt_f1:.4f} | Time: {pt_time:.2f}s")
print(f"  Standard  Stacking   | Acc: {stack_acc:.4f} | F1: {stack_f1:.4f}")
print(f"  Delta (passthrough - standard): {pt_acc - stack_acc:+.4f}")

### 4.10 Label Stacking vs Probability Stacking

When `use_proba=False`, the meta-learner receives hard predicted labels (integers) instead of probability vectors. This dramatically reduces the meta-feature dimensionality: with 4 base models and 10 classes, probability stacking produces 40 meta-features vs just 4 for label stacking.

In [ ]:
def label_vs_proba_stacking(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> pd.DataFrame:
    """Compare label-stacking vs probability-stacking meta-feature strategies.

    Args:
        X_train: Scaled training features.
        y_train: Training labels.
        X_test: Scaled test features.
        y_test: True test labels.

    Returns:
        DataFrame comparing the two approaches.
    """
    rows = []
    for use_proba, strategy_name in [(True, "Proba-stack"), (False, "Label-stack")]:
        stk = StackingEnsemble(
            base_estimators=make_base_estimators(),
            meta_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
            n_folds=N_FOLDS,
            use_proba=use_proba,
        )
        t0 = time.perf_counter()
        stk.fit(X_train, y_train)
        elapsed = time.perf_counter() - t0
        acc = stk.score(X_test, y_test)
        f1  = f1_score(y_test, stk.predict(X_test), average="macro")

        # Count meta-features: proba -> 4 models * 10 classes = 40;
        # label -> 4 models * 1 = 4
        n_meta_feats = (
            len(make_base_estimators()) * len(np.unique(y_train))
            if use_proba
            else len(make_base_estimators())
        )
        rows.append({
            "Strategy": strategy_name,
            "Meta-features": n_meta_feats,
            "Acc": acc,
            "F1 Macro": f1,
            "Time (s)": elapsed,
        })
        print(f"  {strategy_name:<15} | meta_feats={n_meta_feats:>3} | "
              f"Acc={acc:.4f} | F1={f1:.4f} | {elapsed:.1f}s")

    return pd.DataFrame(rows)


print("Label Stacking vs Probability Stacking:")
lv_df = label_vs_proba_stacking(X_train_sc, y_train_full, X_test_sc, y_test)
print()
print(lv_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

### 4.11 Cross-Validation Score Distribution

Single train/test splits can be misleading. We run stratified 5-fold CV on each ensemble strategy and plot the distribution of fold accuracies to assess both mean performance and variance.

In [ ]:
def cv_score_distribution(
    X: np.ndarray,
    y: np.ndarray,
    n_folds: int = 5,
) -> dict:
    """Compute per-fold accuracy for each ensemble strategy.

    Uses StratifiedKFold to ensure balanced class representation in each fold.
    Note: stacking inside CV would require nested CV; here we use a single
    inner split for stacking OOF to keep runtime tractable.

    Args:
        X: Scaled feature matrix.
        y: Target labels.
        n_folds: Number of CV folds.

    Returns:
        Dict mapping strategy name to list of per-fold accuracies.
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    strategy_scores: dict = {
        "Hard Vote": [],
        "Soft Vote": [],
        "Stacking":  [],
        "Blending":  [],
    }

    for fold_num, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[train_idx], X[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        # Hard vote
        hv = HardVotingClassifier(estimators=make_base_estimators())
        hv.fit(X_tr, y_tr)
        strategy_scores["Hard Vote"].append(hv.score(X_va, y_va))

        # Soft vote
        sv = SoftVotingClassifier(estimators=make_base_estimators())
        sv.fit(X_tr, y_tr)
        strategy_scores["Soft Vote"].append(sv.score(X_va, y_va))

        # Stacking (3-fold inner CV to keep runtime manageable)
        meta_inner = LogisticRegression(max_iter=1000, random_state=SEED)
        stk = StackingEnsemble(
            base_estimators=make_base_estimators(),
            meta_estimator=meta_inner,
            n_folds=3,
            use_proba=True,
        )
        stk.fit(X_tr, y_tr)
        strategy_scores["Stacking"].append(stk.score(X_va, y_va))

        # Blending
        meta_bl = LogisticRegression(max_iter=1000, random_state=SEED)
        bl = BlendingEnsemble(
            base_estimators=make_base_estimators(),
            meta_estimator=meta_bl,
            blend_size=BLEND_FRAC,
            use_proba=True,
        )
        bl.fit(X_tr, y_tr)
        strategy_scores["Blending"].append(bl.score(X_va, y_va))

        print(f"  Fold {fold_num} done")

    print("\nCV Summary:")
    for name, scores in strategy_scores.items():
        arr = np.array(scores)
        print(f"  {name:<12} mean={arr.mean():.4f}  std={arr.std():.4f}")
    return strategy_scores


print("Running 5-fold CV for all ensemble strategies (this may take ~30s):")
cv_scores = cv_score_distribution(X_train_full, y_train_full, n_folds=5)

# Box plot of fold accuracy distributions
fig, ax = plt.subplots(figsize=(8, 5))
strategies = list(cv_scores.keys())
data_to_plot = [cv_scores[s] for s in strategies]
bp = ax.boxplot(data_to_plot, labels=strategies, patch_artist=True)
colors_bp = ["#9DC3E6", "#4472C4", "#2E75B6", "#1F4E79"]
for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
ax.set_ylabel("Fold Accuracy")
ax.set_title("5-Fold CV Accuracy Distribution by Ensemble Strategy")
ax.set_ylim(0.90, 1.02)
plt.tight_layout()
plt.show()

### 4.12 Practical Decision Guide — Which Ensemble Strategy to Use?

The choice of ensemble strategy depends on dataset size, compute budget, and acceptable complexity:

| Situation | Recommended strategy | Reason |
|-----------|---------------------|--------|
| Small dataset, tight deadline | Hard / Soft voting | Zero extra CV overhead |
| Calibrated probabilities available | Soft voting | Better than hard vote almost always |
| Training data > 5 000 samples | Stacking (5-fold OOF) | Enough data to train reliable meta-learner |
| Training data < 1 000 samples | Blending with small hold-out | CV overhead not justified |
| Interpretability required | Hard voting | Easier to reason about majority decisions |
| Maximum accuracy needed | Passthrough stacking | Retains original feature signal |

**When ensembles do NOT help:**
- Base models are nearly identical (same algorithm, same hyperparameters)
- Dataset is very small (< 200 samples) — variance dominates
- A single strong model already achieves near-ceiling accuracy
- Inference latency is critical — ensembles are $M 	imes$ slower

In [ ]:
def print_ensemble_decision_guide(
    individual_results: dict,
    ensemble_results: dict,
    n_train: int,
    n_test: int,
) -> None:
    """Print a structured comparison and decision guide for ensemble selection.

    Args:
        individual_results: Dict mapping model name to test accuracy.
        ensemble_results: Dict mapping ensemble name to test accuracy.
        n_train: Number of training samples.
        n_test: Number of test samples.
    """
    best_individual = max(individual_results, key=individual_results.get)
    best_individual_acc = individual_results[best_individual]
    best_ensemble = max(ensemble_results, key=ensemble_results.get)
    best_ensemble_acc = ensemble_results[best_ensemble]

    print("=" * 60)
    print("ENSEMBLE DECISION SUMMARY")
    print("=" * 60)
    print(f"  Dataset     : {n_train} train / {n_test} test samples")
    print(f"  Best single : {best_individual} @ {best_individual_acc:.4f}")
    print(f"  Best ensemble: {best_ensemble} @ {best_ensemble_acc:.4f}")
    gain = best_ensemble_acc - best_individual_acc
    print(f"  Ensemble gain: {gain:+.4f}  ({gain * 100:+.2f}%)")
    print()
    if gain > 0.01:
        print("  Verdict: Ensemble provides meaningful improvement (>1%).")
        print("  Recommended: Stacking or Soft Voting.")
    elif gain > 0.002:
        print("  Verdict: Small but consistent improvement (<1%).")
        print("  Recommended: Soft Voting (lower overhead).")
    else:
        print("  Verdict: Negligible ensemble gain on this dataset.")
        print("  The best single model already captures most of the signal.")
    print()

    # Per-strategy overhead estimate
    print("  Strategy overhead relative to single model training:")
    print(f"    Hard Voting  : ~{len(make_base_estimators())}x  (trains M models once)")
    print(f"    Soft Voting  : ~{len(make_base_estimators())}x  (trains M models once)")
    print(f"    Blending     : ~{len(make_base_estimators())}x + meta-learner training")
    print(f"    Stacking     : ~{N_FOLDS * len(make_base_estimators())}x  "
          f"(trains M models x {N_FOLDS} folds + final refit)")
    print(f"    Passthrough  : ~{N_FOLDS * len(make_base_estimators())}x + larger meta-features")


individual_acc_map = {name: base_accs[name] for name in base_accs}
ensemble_acc_map = {
    "Hard Voting": hard_acc,
    "Soft Voting": soft_acc,
    "Stacking":    stack_acc,
    "Blending":    blend_acc,
}
print_ensemble_decision_guide(
    individual_acc_map,
    ensemble_acc_map,
    n_train=X_train_full.shape[0],
    n_test=X_test.shape[0],
)

### 4.13 Error Correlation Between Base Models

Beyond pairwise disagreement rate, we can compute the *error correlation* matrix: for each pair of models, what fraction of test errors occur on exactly the same samples? High correlation means the models fail together — the ensemble cannot correct those mistakes.

In [ ]:
def error_correlation_matrix(
    preds_dict: dict,
    y_true: np.ndarray,
) -> np.ndarray:
    """Compute pairwise error-correlation matrix for a set of classifiers.

    For each pair (i, j), the correlation is the fraction of all test errors
    that are shared by both models (Jaccard similarity of error sets).

    Args:
        preds_dict: Dict mapping model name to predicted label array.
        y_true: Ground truth labels.

    Returns:
        Square correlation matrix of shape (n_models, n_models).
    """
    names = list(preds_dict.keys())
    n = len(names)
    corr = np.zeros((n, n))
    # Boolean error masks: True where the model is wrong
    error_masks = {name: (preds_dict[name] != y_true) for name in names}
    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            mask_i = error_masks[ni]
            mask_j = error_masks[nj]
            intersection = (mask_i & mask_j).sum()
            union = (mask_i | mask_j).sum()
            corr[i, j] = intersection / union if union > 0 else 0.0
    return corr


# Collect test predictions from fitted base models
base_preds_for_corr: dict = {}
for name, clf in make_base_estimators():
    clf.fit(X_train_sc, y_train_full)
    base_preds_for_corr[name] = clf.predict(X_test_sc)

corr_matrix = error_correlation_matrix(base_preds_for_corr, y_test)
model_names_corr = list(base_preds_for_corr.keys())
n_models_corr = len(model_names_corr)

# Print error counts per model
print("Error counts per base model:")
for name, preds in base_preds_for_corr.items():
    n_errors = (preds != y_test).sum()
    print(f"  {name:<8}: {n_errors} errors / {len(y_test)} samples "
          f"({n_errors / len(y_test):.2%} error rate)")

print()
print("Pairwise error correlation (Jaccard similarity of error sets):")
print("  (1.0 = always fail together; 0.0 = never fail on same sample)")
for i, ni in enumerate(model_names_corr):
    for j, nj in enumerate(model_names_corr):
        if i < j:
            print(f"  {ni} & {nj}: {corr_matrix[i, j]:.3f}")

# Heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr_matrix, cmap="Reds", vmin=0, vmax=1)
ax.set_xticks(range(n_models_corr))
ax.set_yticks(range(n_models_corr))
ax.set_xticklabels(model_names_corr)
ax.set_yticklabels(model_names_corr)
plt.colorbar(im, ax=ax, label="Error Jaccard Similarity")
ax.set_title("Error Correlation Matrix (Jaccard)")
for i in range(n_models_corr):
    for j in range(n_models_corr):
        ax.text(j, i, f"{corr_matrix[i, j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if corr_matrix[i, j] > 0.6 else "black")
plt.tight_layout()
plt.show()
print("Low Jaccard correlation = high error diversity = strong ensemble candidate.")

---

## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Hard voting** aggregates predicted labels by plurality; **soft voting** averages class probabilities — soft voting is almost always better because it uses each model's confidence, not just which class wins.
- **Stacking** trains a meta-learner on *out-of-fold (OOF)* predictions from cross-validation, eliminating the data leakage that arises when base models predict on their own training data. **Blending** is a faster but data-wasteful alternative using a held-out blend set.
- **Diversity drives ensemble gain.** An ensemble of $n$ identical models performs no better than a single model; heterogeneous classifiers with different inductive biases (tree, distance, probabilistic, linear) make complementary errors and combine to outperform any individual.
- **Diminishing returns** appear quickly — a well-chosen ensemble of 3–4 diverse models typically captures most of the available benefit; adding a 5th or 6th rarely justifies the overhead.
- **The meta-learner matters less than expected** — a simple Logistic Regression on OOF probability features usually performs as well as a more complex level-1 model, because the OOF features already encode well-calibrated discriminative signals.

### What's Next

→ **02-10 (Model Comparison & Algorithm Selection)** trains *all* algorithms from Module 02 on the same Digits dataset under identical conditions and develops a decision framework for choosing the right algorithm based on data size, dimensionality, interpretability, and latency.